KNN Fraud Detection
### Algoritma: K-Nearest Neighbor (KNN) Tanpa Library
**Dataset:** Credit Card Fraud Detection  
**Metode:** KNN Manual (tanpa scikit-learn / library ML)  
**Evaluasi:** Accuracy, Precision, Recall, F1-Score, Confusion Matrix

---
> ⚠️ **Semua proses dilakukan manual** — tidak menggunakan sklearn, scipy, atau library ML lainnya.


## STEP 1 — Upload Dataset

In [1]:
import os

print(os.path.exists("sample_data.csv"))

True


## STEP 2 — Baca Dataset

**Penjelasan:**  
Kode ini membaca file CSV `sample_data.csv` secara manual baris per baris menggunakan modul `csv` bawaan Python (bukan pandas).  
Setiap baris dikonversi menjadi list angka, lalu header disimpan terpisah.


In [2]:
import csv

# Baca file CSV secara manual
with open("sample_data.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)       # Baris pertama = nama kolom
    raw_data = []
    for row in reader:
        raw_data.append([float(x) for x in row])

print("Kolom:", header)
print("Jumlah data:", len(raw_data))
print("Contoh baris pertama:", raw_data[0][:5], "...")


Kolom: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Jumlah data: 492
Contoh baris pertama: [153863.0, 153863.0, 153863.0, 153863.0, 153863.0] ...


## STEP 3 — Bersihkan Data & Lihat Distribusi Kelas

**Penjelasan:**  
Dataset memiliki 1 baris corrupt (row 0) di mana kolom V1–V24 semuanya bernilai sama dengan kolom Time (153863.0).  
Baris ini dihapus sebelum proses training agar tidak merusak model.

Setelah dibersihkan, distribusi kelas dihitung manual untuk memastikan dataset seimbang.


In [3]:
# ── Deteksi & hapus baris corrupt ──────────────────────────────────────────
# Row corrupt: jika nilai V1 == nilai Time (kolom 0 == kolom 1)
data = []
n_corrupt = 0
for row in raw_data:
    if row[0] == row[1]:   # Time == V1 → baris rusak
        n_corrupt += 1
    else:
        data.append(row)

print(f"Baris corrupt dihapus : {n_corrupt}")
print(f"Data bersih tersisa   : {len(data)}")

# ── Distribusi kelas manual ─────────────────────────────────────────────────
count_class = {}
for row in data:
    label = int(row[-1])
    count_class[label] = count_class.get(label, 0) + 1

print("\nDistribusi Kelas:")
for k, v in sorted(count_class.items()):
    nama = "Normal (Non-Fraud)" if k == 0 else "Fraud"
    print(f"  Class {k} — {nama}: {v} data")


Baris corrupt dihapus : 1
Data bersih tersisa   : 491

Distribusi Kelas:
  Class 0 — Normal (Non-Fraud): 245 data
  Class 1 — Fraud: 246 data


## STEP 4 — Pisahkan Fitur & Target + Normalisasi Min-Max Manual

**Penjelasan Matematika Normalisasi Min-Max:**

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

- Nilai hasil normalisasi berada di rentang **[0, 1]**
- Penting dilakukan agar fitur dengan skala besar (seperti `Time`, `Amount`) tidak mendominasi perhitungan jarak

**X** = semua kolom kecuali `Class` (Time, V1–V28, Amount)  
**y** = kolom `Class` (0 = Normal, 1 = Fraud)


In [4]:
# ── Pisahkan fitur (X) dan target (y) ──────────────────────────────────────
X = [row[:-1] for row in data]
y = [int(row[-1]) for row in data]

n_rows    = len(X)
n_cols    = len(X[0])

print(f"Jumlah data  : {n_rows}")
print(f"Jumlah fitur : {n_cols}")

# ── Normalisasi Min-Max Manual ──────────────────────────────────────────────
min_vals = [min(X[r][c] for r in range(n_rows)) for c in range(n_cols)]
max_vals = [max(X[r][c] for r in range(n_rows)) for c in range(n_cols)]

X_norm = []
for r in range(n_rows):
    row_norm = []
    for c in range(n_cols):
        denom = max_vals[c] - min_vals[c]
        if denom != 0:
            row_norm.append((X[r][c] - min_vals[c]) / denom)
        else:
            row_norm.append(0.0)
    X_norm.append(row_norm)

print("\nNormalisasi selesai. Contoh baris pertama (5 fitur pertama):")
print([round(v, 4) for v in X_norm[0][:5]])


Jumlah data  : 491
Jumlah fitur : 30

Normalisasi selesai. Contoh baris pertama (5 fitur pertama):
[0.1968, 0.9213, 0.3962, 0.9376, 0.2406]


## STEP 5 — Split Data Training & Testing Manual (80:20)

**Penjelasan:**  
Data dibagi secara manual tanpa acak (sequential split):
- **80%** pertama → data training (digunakan untuk melatih model)
- **20%** sisanya → data testing (digunakan untuk menguji model)

Pembagian dilakukan pada **data yang sudah dinormalisasi** (`X_norm`).


In [5]:
split_index = int(n_rows * 0.8)

X_train = X_norm[:split_index]
X_test  = X_norm[split_index:]

y_train = y[:split_index]
y_test  = y[split_index:]

print(f"Total data    : {n_rows}")
print(f"Data training : {len(X_train)} ({round(len(X_train)/n_rows*100)}%)")
print(f"Data testing  : {len(X_test)}  ({round(len(X_test)/n_rows*100)}%)")


Total data    : 491
Data training : 392 (80%)
Data testing  : 99  (20%)


## STEP 6 — Rumus Euclidean Distance (Manual, Tanpa Library)

**Penjelasan Matematika:**

$$d(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Rumus ini menghitung **jarak lurus** antara dua titik di ruang berdimensi-n.  
- Semakin **kecil** jarak → data semakin **mirip**  
- Semakin **besar** jarak → data semakin **berbeda**

> ℹ️ Fungsi ini **tidak menggunakan `import math`** — menggunakan `** 0.5` sebagai pengganti `math.sqrt()`.


In [6]:
def euclidean_distance(a, b):
    """
    Hitung Euclidean Distance antara vektor a dan b secara manual.
    Rumus: sqrt( sum( (a[i] - b[i])^2 ) )
    Tanpa library math — menggunakan ** 0.5
    """
    total = 0
    for i in range(len(a)):
        total += (a[i] - b[i]) ** 2
    return total ** 0.5   # pengganti math.sqrt()

# ── Uji fungsi ──────────────────────────────────────────────────────────────
a = [0.1, 0.5, 0.3]
b = [0.4, 0.2, 0.9]
print("Uji Euclidean Distance:")
print(f"  a = {a}")
print(f"  b = {b}")
print(f"  jarak = {round(euclidean_distance(a, b), 6)}")


Uji Euclidean Distance:
  a = [0.1, 0.5, 0.3]
  b = [0.4, 0.2, 0.9]
  jarak = 0.734847


## STEP 7 — Algoritma KNN Manual

**Cara Kerja KNN Langkah demi Langkah:**

1. **Hitung jarak** antara data testing ke semua data training menggunakan Euclidean Distance
2. **Urutkan** jarak dari yang terkecil ke terbesar
3. **Ambil K tetangga terdekat** (K data dengan jarak terkecil)
4. **Voting mayoritas** — kelas yang paling banyak muncul di antara K tetangga = hasil prediksi
5. **Kembalikan kelas** hasil voting sebagai prediksi akhir


In [7]:
def knn_predict(test_row, X_train, y_train, k):
    """
    Prediksi kelas untuk satu data menggunakan KNN manual.
    Langkah:
      1. Hitung jarak ke semua training data
      2. Urutkan dari jarak terkecil
      3. Ambil k tetangga terdekat
      4. Voting mayoritas
      5. Return kelas dengan suara terbanyak
    """
    # Langkah 1: Hitung semua jarak
    distances = []
    for i in range(len(X_train)):
        dist = euclidean_distance(test_row, X_train[i])
        distances.append((dist, y_train[i]))

    # Langkah 2: Urutkan berdasarkan jarak (ascending)
    distances.sort(key=lambda x: x[0])

    # Langkah 3: Ambil K tetangga terdekat
    neighbors = distances[:k]

    # Langkah 4: Voting mayoritas
    votes = {}
    for _, label in neighbors:
        votes[label] = votes.get(label, 0) + 1

    # Langkah 5: Return kelas dengan suara terbanyak
    return max(votes, key=votes.get)

print("Fungsi KNN berhasil didefinisikan ✓")


Fungsi KNN berhasil didefinisikan ✓


## STEP 8 — Tentukan Nilai K

**Penjelasan:**  
Nilai **K = 3** artinya model mengambil **3 tetangga terdekat** untuk melakukan voting.

- K ganjil dipilih agar tidak terjadi seri saat voting (2 vs 2)
- K terlalu kecil → model sensitif terhadap noise
- K terlalu besar → model terlalu umum (underfitting)


In [8]:
k = 3
print(f"Nilai K yang digunakan: {k}")
print(f"Artinya: model akan memilih {k} data training terdekat untuk setiap prediksi.")


Nilai K yang digunakan: 3
Artinya: model akan memilih 3 data training terdekat untuk setiap prediksi.


## STEP 9 — Prediksi Data Testing

**Penjelasan:**  
Setiap baris data testing akan diprediksi satu per satu menggunakan fungsi `knn_predict`.  
Data lama (X_test yang berasal dari dataset asli) digunakan di sini.

> Proses ini mungkin memakan waktu beberapa menit karena dilakukan manual tanpa optimasi.


In [9]:
print("Memulai prediksi data testing...")
print(f"Jumlah data testing: {len(X_test)}")
print("Mohon tunggu...\n")

predictions = []
for i, row in enumerate(X_test):
    pred = knn_predict(row, X_train, y_train, k)
    predictions.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  Progress: {i+1}/{len(X_test)} data selesai diprediksi...")

print(f"\n✅ Prediksi selesai! Total prediksi: {len(predictions)}")


Memulai prediksi data testing...
Jumlah data testing: 99
Mohon tunggu...

  Progress: 10/99 data selesai diprediksi...
  Progress: 20/99 data selesai diprediksi...
  Progress: 30/99 data selesai diprediksi...
  Progress: 40/99 data selesai diprediksi...
  Progress: 50/99 data selesai diprediksi...
  Progress: 60/99 data selesai diprediksi...
  Progress: 70/99 data selesai diprediksi...
  Progress: 80/99 data selesai diprediksi...
  Progress: 90/99 data selesai diprediksi...

✅ Prediksi selesai! Total prediksi: 99


## STEP 10 — Evaluasi: Accuracy

**Rumus Matematika:**

$$Accuracy = \frac{\text{Jumlah prediksi benar}}{\text{Total data testing}} \times 100\%$$

Mengukur seberapa banyak prediksi yang tepat dari keseluruhan data testing.


In [10]:
correct = 0
for i in range(len(y_test)):
    if predictions[i] == y_test[i]:
        correct += 1

accuracy = (correct / len(y_test)) * 100

print("=" * 40)
print("       HASIL EVALUASI — ACCURACY")
print("=" * 40)
print(f"  Prediksi benar : {correct}")
print(f"  Total testing  : {len(y_test)}")
print(f"  Accuracy       : {round(accuracy, 2)}%")
print("=" * 40)


       HASIL EVALUASI — ACCURACY
  Prediksi benar : 94
  Total testing  : 99
  Accuracy       : 94.95%


## STEP 11 — Confusion Matrix Manual

**Penjelasan:**

|  | Prediksi Fraud (1) | Prediksi Normal (0) |
|---|---|---|
| **Aktual Fraud (1)** | TP (True Positive) | FN (False Negative) |
| **Aktual Normal (0)** | FP (False Positive) | TN (True Negative) |

- **TP** = Fraud, diprediksi Fraud ✅  
- **TN** = Normal, diprediksi Normal ✅  
- **FP** = Normal, diprediksi Fraud ❌ (False Alarm)  
- **FN** = Fraud, diprediksi Normal ❌ (Berbahaya!)


In [11]:
TP = TN = FP = FN = 0

for i in range(len(y_test)):
    actual = y_test[i]
    pred   = predictions[i]
    if   actual == 1 and pred == 1: TP += 1
    elif actual == 0 and pred == 0: TN += 1
    elif actual == 0 and pred == 1: FP += 1
    elif actual == 1 and pred == 0: FN += 1

print("=" * 40)
print("        CONFUSION MATRIX")
print("=" * 40)
print(f"  TP (Fraud → Fraud)   : {TP}")
print(f"  TN (Normal → Normal) : {TN}")
print(f"  FP (Normal → Fraud)  : {FP}  ← False Alarm")
print(f"  FN (Fraud → Normal)  : {FN}  ← Berbahaya!")
print("=" * 40)
print()
print("Tabel Confusion Matrix:")
print(f"  {'':20s} | Pred Fraud | Pred Normal")
print(f"  {'Aktual Fraud':20s} |   {TP:5d}    |   {FN:5d}")
print(f"  {'Aktual Normal':20s} |   {FP:5d}    |   {TN:5d}")


        CONFUSION MATRIX
  TP (Fraud → Fraud)   : 51
  TN (Normal → Normal) : 43
  FP (Normal → Fraud)  : 1  ← False Alarm
  FN (Fraud → Normal)  : 4  ← Berbahaya!

Tabel Confusion Matrix:
                       | Pred Fraud | Pred Normal
  Aktual Fraud         |      51    |       4
  Aktual Normal        |       1    |      43


## STEP 12 — Precision, Recall, F1-Score Manual

**Rumus Matematika:**

$$Precision = \frac{TP}{TP + FP}$$

$$Recall = \frac{TP}{TP + FN}$$

$$F1\text{-}Score = \frac{2 \times Precision \times Recall}{Precision + Recall}$$

- **Precision** → dari semua yang diprediksi Fraud, berapa yang benar-benar Fraud?
- **Recall** → dari semua Fraud yang ada, berapa yang berhasil terdeteksi?
- **F1-Score** → keseimbangan antara Precision dan Recall


In [12]:
precision = (TP / (TP + FP)) if (TP + FP) != 0 else 0
recall    = (TP / (TP + FN)) if (TP + FN) != 0 else 0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) != 0 else 0

print("=" * 40)
print("    HASIL EVALUASI LENGKAP")
print("=" * 40)
print(f"  Accuracy  : {round(accuracy,  4) * 100:.2f}%")
print(f"  Precision : {round(precision, 4)}")
print(f"  Recall    : {round(recall,    4)}")
print(f"  F1-Score  : {round(f1,        4)}")
print("=" * 40)


    HASIL EVALUASI LENGKAP
  Accuracy  : 9494.95%
  Precision : 0.9808
  Recall    : 0.9273
  F1-Score  : 0.9533


## STEP 13 — Visualisasi Dataset (ASCII, Tanpa Library)

**Penjelasan:**  
Karena tidak boleh menggunakan library (matplotlib), visualisasi dilakukan menggunakan **ASCII art** manual di terminal/output.

Fitur yang divisualisasikan: **V1 (sumbu X)** vs **V2 (sumbu Y)** — dipilih karena memiliki variasi terbesar antar kelas.

**Decision Boundary:**  
KNN tidak memiliki garis batas eksplisit, namun secara konseptual:  
> Batas keputusan KNN terbentuk dari **area Voronoi** — setiap titik testing diklasifikasikan berdasarkan K tetangga terdekatnya.  
> Pada plot ASCII ini, perbedaan posisi titik `0` (Normal) dan `F` (Fraud) menggambarkan pemisahan kelas secara visual.


In [13]:
# ── Visualisasi ASCII Scatter Plot ──────────────────────────────────────────
# Ambil fitur V1 (index 1) dan V2 (index 2) dari X_norm

v1_norm = [X_norm[i][1] for i in range(len(X_norm))]
v2_norm = [X_norm[i][2] for i in range(len(X_norm))]
labels  = y

# Ukuran canvas ASCII
WIDTH  = 60
HEIGHT = 20

canvas = [['.' for _ in range(WIDTH)] for _ in range(HEIGHT)]

for i in range(len(v1_norm)):
    col = int(v1_norm[i] * (WIDTH  - 1))
    row = int((1 - v2_norm[i]) * (HEIGHT - 1))   # balik sumbu Y
    col = max(0, min(WIDTH  - 1, col))
    row = max(0, min(HEIGHT - 1, row))
    if labels[i] == 1:
        canvas[row][col] = 'F'    # Fraud
    else:
        if canvas[row][col] != 'F':   # jangan timpa Fraud
            canvas[row][col] = '0'    # Normal

print("=" * (WIDTH + 4))
print(" SCATTER PLOT: V1 (X) vs V2 (Y)")
print(" Legend: F = Fraud (1)  |  0 = Normal  |  . = kosong")
print("=" * (WIDTH + 4))
print(f"  ^")
for row in canvas:
    print(f"  | {''.join(row)}")
print(f"  +{'─'*WIDTH}>")
print(f"    V1 (ternormalisasi) →")
print()

# Hitung ringkasan
n_fraud  = labels.count(1)
n_normal = labels.count(0)
print(f"Total titik: {len(labels)}  |  Normal (0): {n_normal}  |  Fraud (F): {n_fraud}")


 SCATTER PLOT: V1 (X) vs V2 (Y)
 Legend: F = Fraud (1)  |  0 = Normal  |  . = kosong
  ^
  | FFFFF.F.....................................................
  | ........FFF.................................................
  | ...........F.FFFF..............F............................
  | ..................FFFF......................................
  | ......................FF.FF.................................
  | .................F..F.F.FFF..................FF.............
  | ...........................FFF....F.FF.......F.F............
  | ............................FFFF.F.FF.FF.F...F.F..FF........
  | ................................F...F...FF.FF0FFFFFF........
  | ................................F............FFFFFFFFFFFF...
  | .......................................FFF.F.FF.FFF0FFFFFF..
  | ...........................................FF.FFFFFFFFFFFF0.
  | .............................................FFFFFFFFFF0FF00
  | .................................................F0FFF00000.
 

## STEP 14 — Decision Boundary (Konsep Visual Manual)

**Penjelasan Konsep Decision Boundary KNN:**

KNN tidak membentuk garis/kurva batas yang eksplisit seperti SVM atau Logistic Regression.  
Batas keputusan KNN bersifat **non-linear** dan terbentuk secara implisit dari posisi data training.

**Cara kerja:**  
Untuk setiap titik di ruang fitur, KNN mencari K tetangga terdekat dari data training.  
Titik-titik dengan kelas mayoritas yang berbeda akan membentuk **batas wilayah** (decision region).

Visualisasi di bawah mensimulasikan decision boundary di ruang 2D (V1 vs V2) dengan resolusi grid kasar.


In [14]:
# ── Decision Boundary Simulation (Grid 2D, ASCII) ───────────────────────────
# Gunakan subset kecil training data agar tidak terlalu lama
# Ambil 50 data training pertama untuk demo

X_db    = [[X_norm[i][1], X_norm[i][2]] for i in range(min(80, len(X_train)))]
y_db    = y_train[:len(X_db)]

def knn_2d(test_row, X_tr, y_tr, k=3):
    dists = []
    for i in range(len(X_tr)):
        d = ((test_row[0]-X_tr[i][0])**2 + (test_row[1]-X_tr[i][1])**2) ** 0.5
        dists.append((d, y_tr[i]))
    dists.sort(key=lambda x: x[0])
    votes = {}
    for _, lbl in dists[:k]:
        votes[lbl] = votes.get(lbl, 0) + 1
    return max(votes, key=votes.get)

COLS = 50
ROWS = 18
boundary_canvas = []

for r in range(ROWS):
    row_str = ""
    for c in range(COLS):
        x_val = c / (COLS - 1)
        y_val = 1 - r / (ROWS - 1)
        pred = knn_2d([x_val, y_val], X_db, y_db, k=3)
        row_str += ("▓" if pred == 1 else "░")
    boundary_canvas.append(row_str)

print("=" * (COLS + 4))
print(" DECISION BOUNDARY — KNN (K=3)")
print(" ▓ = Wilayah Fraud  |  ░ = Wilayah Normal")
print("=" * (COLS + 4))
print("  ^")
for row in boundary_canvas:
    print(f"  | {row}")
print(f"  +{'─'*COLS}>")
print(f"    V1 →")
print()
print("Keterangan:")
print("  ▓ (gelap) = area di mana model memprediksi FRAUD")
print("  ░ (terang) = area di mana model memprediksi NORMAL")
print("  Batas antara ▓ dan ░ = Decision Boundary KNN")


 DECISION BOUNDARY — KNN (K=3)
 ▓ = Wilayah Fraud  |  ░ = Wilayah Normal
  ^
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░▓▓▓▓▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░▓▓▓
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░
  | ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

## STEP 15 — Prediksi Data Baru (Manual, Sudah Ternormalisasi)

**Penjelasan:**  
Data baru yang dimasukkan **harus dinormalisasi terlebih dahulu** menggunakan nilai min/max dari data training.  
Ini penting agar skala data baru konsisten dengan data training.

Data baru di bawah merepresentasikan sebuah transaksi hipotetis dengan nilai yang bervariasi.

> ℹ️ `data_baru_raw` berisi nilai asli → lalu dinormalisasi → baru diprediksi.


In [15]:
# ── Data baru (30 fitur: Time, V1-V28, Amount) dalam nilai ASLI ─────────────
data_baru_raw = [
    #  Time      V1       V2       V3       V4       V5
    50000,    -1.5,     2.1,    -0.8,     0.5,    -1.2,
    #  V6       V7       V8       V9      V10      V11
       0.3,    -0.7,     0.9,    -0.4,     1.1,    -0.3,
    #  V12      V13      V14      V15      V16      V17
      -0.6,     0.8,    -1.0,     0.2,    -0.5,     1.3,
    #  V18      V19      V20      V21      V22      V23
       0.1,    -0.9,     0.4,    -0.2,     0.7,    -0.1,
    #  V24      V25      V26      V27      V28    Amount
       0.6,    -0.3,     0.5,    -0.8,     0.2,   120.0
]

# ── Normalisasi data baru menggunakan min/max dari data TRAINING ─────────────
data_baru_norm = []
for c in range(len(data_baru_raw)):
    denom = max_vals[c] - min_vals[c]
    if denom != 0:
        val_norm = (data_baru_raw[c] - min_vals[c]) / denom
    else:
        val_norm = 0.0
    # Clip ke [0, 1] untuk antisipasi data di luar range training
    val_norm = max(0.0, min(1.0, val_norm))
    data_baru_norm.append(val_norm)

# ── Prediksi ─────────────────────────────────────────────────────────────────
hasil = knn_predict(data_baru_norm, X_train, y_train, k)

print("=" * 45)
print("       PREDIKSI DATA BARU")
print("=" * 45)
print(f"  Nilai raw (5 pertama) : {data_baru_raw[:5]}")
print(f"  Setelah normalisasi   : {[round(v,4) for v in data_baru_norm[:5]]}")
print()
if hasil == 1:
    print("  🚨 HASIL PREDIKSI : FRAUD (1)")
    print("  ⚠️  Transaksi ini terindikasi MENCURIGAKAN!")
else:
    print("  ✅ HASIL PREDIKSI : NORMAL (0)")
    print("  👍 Transaksi ini terindikasi AMAN.")
print("=" * 45)


       PREDIKSI DATA BARU
  Nilai raw (5 pertama) : [50000, -1.5, 2.1, -0.8, 0.5]
  Setelah normalisasi   : [0.2886, 0.8859, 0.4351, 0.8773, 0.2683]

  ✅ HASIL PREDIKSI : NORMAL (0)
  👍 Transaksi ini terindikasi AMAN.


## Kesimpulan

> *"Pada penelitian ini saya menggunakan algoritma K-Nearest Neighbor (KNN) tanpa menggunakan library machine learning apapun. Dataset Credit Card Fraud dibagi menjadi 80% data training dan 20% data testing. Sebelum proses training, dilakukan pembersihan data corrupt dan normalisasi Min-Max secara manual.*
>
> *Proses klasifikasi dilakukan dengan menghitung jarak Euclidean antara data testing dan seluruh data training, kemudian memilih 3 tetangga terdekat (K=3) dan voting mayoritas.*
>
> *Hasil klasifikasi dievaluasi menggunakan Accuracy, Precision, Recall, dan F1-Score yang semuanya dihitung manual. Decision Boundary divisualisasikan dalam bentuk grid ASCII 2D menggunakan fitur V1 dan V2.*
>
> *Prediksi terhadap data baru dilakukan dengan normalisasi terlebih dahulu menggunakan parameter min/max dari data training."*


